# Notebook 06 - Feature Engineering with Pandas

**Topic:** Data Processing with Pandas  
**Subtopic:** Feature Engineering

---

## What you will learn

- What feature engineering is and why it matters for machine learning
- How to create features from numeric columns (scaling, binning, ratios)
- How to create features from datetime columns
- How to create lag and rolling window features for time series
- How to create interaction and polynomial features
- How to select features using correlation and variance
- How to build a reproducible feature engineering pipeline

---

## What is Feature Engineering?

Feature engineering is the process of creating new input variables (features) from raw data to improve the predictive power of a machine learning model. A model cannot learn what it cannot see. If a pattern exists only in the interaction between two columns, and you never compute that interaction, the model will miss it entirely.

Raw data rarely comes in a form that is directly useful for a model. Feature engineering bridges that gap.

---

## Prerequisites

```bash
pip install pandas numpy matplotlib scikit-learn
```

---

## Section 1 - Imports and Dataset Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.preprocessing import StandardScaler, MinMaxScaler

np.random.seed(42)
os.makedirs("output", exist_ok=True)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Ready.")

In [ ]:
# We use a realistic e-commerce orders dataset for this notebook
# It combines customer, product, and transaction data

n = 500

np.random.seed(42)
order_dates = pd.date_range(start="2022-01-01", end="2024-06-30", periods=n)

categories = np.random.choice(
    ["Electronics", "Books", "Clothing", "Home & Kitchen", "Sports"],
    p=[0.30, 0.20, 0.20, 0.15, 0.15],
    size=n
)

# Unit price varies by category
price_params = {
    "Electronics"   : (120, 60),
    "Books"         : (28, 10),
    "Clothing"      : (55, 30),
    "Home & Kitchen": (75, 45),
    "Sports"        : (45, 25),
}

unit_prices = np.array([
    np.clip(np.random.normal(*price_params[c]), 5, 500)
    for c in categories
]).round(2)

quantities  = np.random.choice([1, 2, 3, 4, 5], p=[0.55, 0.25, 0.10, 0.06, 0.04], size=n)
discount_pct = np.random.choice([0, 5, 10, 15, 20, 25, 30], p=[0.50, 0.12, 0.12, 0.10, 0.08, 0.05, 0.03], size=n)

total_price = (unit_prices * quantities).round(2)
final_price = (total_price * (1 - discount_pct / 100)).round(2)

return_prob = np.minimum(0.25, final_price / 2000)
is_returned  = (np.random.rand(n) < return_prob).astype(int)

customer_ids = np.random.randint(1000, 1200, size=n)  # 200 unique customers

df = pd.DataFrame({
    "order_id"        : [f"ORD-{i:04d}" for i in range(1, n + 1)],
    "customer_id"     : customer_ids,
    "order_date"      : np.sort(order_dates),
    "product_category": categories,
    "unit_price_usd"  : unit_prices,
    "quantity"        : quantities,
    "total_price_usd" : total_price,
    "discount_pct"    : discount_pct,
    "final_price_usd" : final_price,
    "is_returned"     : is_returned,
})

print(f"Dataset shape: {df.shape}")
df.head()

---

## Section 2 - Numeric Features: Scaling

Most ML algorithms are sensitive to the scale of input features. A feature with values in the range 0-100,000 will dominate a feature with values in the range 0-1, even if the smaller feature is more predictive.

### Two standard approaches

**Standard scaling (Z-score):** Transforms the column to have mean = 0 and standard deviation = 1. Use when the data is approximately normally distributed and you want to preserve the shape.

```
z = (x - mean) / std
```

**Min-Max scaling:** Compresses all values into the range [0, 1]. Use when you need a hard boundary (e.g. for neural networks or when you know the min/max meaningful values).

```
x_scaled = (x - min) / (max - min)
```

In [ ]:
numeric_cols = ["unit_price_usd", "quantity", "total_price_usd", "final_price_usd"]

# Standard scaling
scaler_std = StandardScaler()
scaled_std = scaler_std.fit_transform(df[numeric_cols])
df_std = pd.DataFrame(scaled_std, columns=[f"{c}_zscore" for c in numeric_cols])

# Min-Max scaling
scaler_mm = MinMaxScaler()
scaled_mm = scaler_mm.fit_transform(df[numeric_cols])
df_mm = pd.DataFrame(scaled_mm, columns=[f"{c}_minmax" for c in numeric_cols])

print("Standard scaled (mean should be ~0, std should be ~1):")
print(df_std.describe().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3)

df["final_price_usd"].plot(kind="hist", bins=40, ax=axes[0], color="#5b8dd9")
axes[0].set_title("Original")
axes[0].set_xlabel("final_price_usd")

df_std["final_price_usd_zscore"].plot(kind="hist", bins=40, ax=axes[1], color="#2a5fc1")
axes[1].set_title("Z-score scaled")
axes[1].set_xlabel("Z-score")

df_mm["final_price_usd_minmax"].plot(kind="hist", bins=40, ax=axes[2], color="#1a3f8a")
axes[2].set_title("Min-Max scaled")
axes[2].set_xlabel("[0, 1]")

plt.tight_layout()
plt.savefig("output/fe_scaling.png", dpi=120)
plt.show()

print("Shape is preserved. Only the axis scale changes.")

**Important:** Always fit the scaler on the training set only, then use `transform()` (not `fit_transform()`) on the test set. Fitting on the full dataset leaks test information into training.

---

## Section 3 - Numeric Features: Binning

Binning converts a continuous numeric column into ordered categories. This is useful when:
- The relationship between the variable and the target is non-linear (e.g. very low and very high prices both return)
- You want to reduce sensitivity to outliers
- You want interpretable segments (age groups, price tiers)

In [ ]:
# Fixed-width bins (you define the boundaries)
df["price_tier"] = pd.cut(
    df["final_price_usd"],
    bins=[0, 30, 75, 150, 300, np.inf],
    labels=["Budget", "Economy", "Standard", "Premium", "Luxury"]
)

print("Fixed-width bins (price_tier):")
print(df["price_tier"].value_counts().sort_index().to_string())

In [ ]:
# Quantile bins (equal-frequency: each bin holds the same number of records)
df["price_quartile"] = pd.qcut(
    df["final_price_usd"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print("Quantile bins (price_quartile):")
print(df["price_quartile"].value_counts().sort_index().to_string())

# Verify that each bin has approximately the same count
print("\nReturn rate by price quartile:")
print(
    df.groupby("price_quartile", observed=True)["is_returned"]
    .mean().round(3).to_string()
)

---

## Section 4 - Numeric Features: Ratios and Interactions

Ratios and products between columns often capture relationships that neither column alone can express.

In [ ]:
# Ratio features
df["discount_ratio"]      = (df["discount_pct"] / 100).round(4)
df["saving_usd"]          = (df["total_price_usd"] - df["final_price_usd"]).round(2)
df["price_per_unit"]      = (df["final_price_usd"] / df["quantity"]).round(2)

# Interaction feature: high quantity + high discount is a specific pattern
df["quantity_x_discount"] = df["quantity"] * df["discount_pct"]

# Log transform: useful when a column is heavily right-skewed
# log(1 + x) handles zeros without producing -inf
df["log_final_price"] = np.log1p(df["final_price_usd"]).round(4)

print(df[["final_price_usd", "saving_usd", "price_per_unit", "log_final_price"]].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2)

df["final_price_usd"].plot(kind="hist", bins=40, ax=axes[0], color="#5b8dd9")
axes[0].set_title("final_price_usd (skewed right)")

df["log_final_price"].plot(kind="hist", bins=40, ax=axes[1], color="#2a5fc1")
axes[1].set_title("log(1 + final_price_usd) (more symmetric)")

plt.tight_layout()
plt.savefig("output/fe_log_transform.png", dpi=120)
plt.show()

---

## Section 5 - Datetime Features

Datetime columns are rich with potential features. Seasonality, day-of-week effects, and recency all live inside a single timestamp.

In [ ]:
# Calendar components
df["order_year"]     = df["order_date"].dt.year
df["order_month"]    = df["order_date"].dt.month
df["order_quarter"]  = df["order_date"].dt.quarter
df["order_dayofweek"]= df["order_date"].dt.dayofweek   # 0 = Monday, 6 = Sunday
df["order_dayofyear"]= df["order_date"].dt.dayofyear
df["is_weekend"]     = (df["order_date"].dt.dayofweek >= 5).astype(int)

# Recency: days since the most recent date in the dataset
reference_date = df["order_date"].max()
df["days_since_order"] = (reference_date - df["order_date"]).dt.days

print("Weekend orders vs weekday orders:")
print(df.groupby("is_weekend")["is_returned"].agg(["mean", "count"]).round(3).to_string())

In [ ]:
# Cyclical encoding: month 1 and month 12 are close in time but far in numeric space
# Sine and cosine encoding wraps the cycle correctly

df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12).round(4)
df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12).round(4)
df["dow_sin"]   = np.sin(2 * np.pi * df["order_dayofweek"] / 7).round(4)
df["dow_cos"]   = np.cos(2 * np.pi * df["order_dayofweek"] / 7).round(4)

print("Cyclical features for month (first 3 rows):")
print(df[["order_month", "month_sin", "month_cos"]].head(3).to_string())

fig, ax = plt.subplots(figsize=(6, 6))
months = np.arange(1, 13)
ax.scatter(
    np.sin(2 * np.pi * months / 12),
    np.cos(2 * np.pi * months / 12),
    s=200, color="#2a5fc1"
)
for m in months:
    ax.annotate(
        str(m),
        (np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)),
        textcoords="offset points", xytext=(8, 4), fontsize=10
    )
ax.set_title("Cyclical encoding of month\n(months 1 and 12 are neighbours on the circle)")
ax.set_xlabel("month_sin")
ax.set_ylabel("month_cos")
plt.tight_layout()
plt.savefig("output/fe_cyclical_encoding.png", dpi=120)
plt.show()

---

## Section 6 - Customer-Level Aggregated Features

One of the most powerful techniques in e-commerce and other transactional domains is building features that describe a customer's history up to each transaction. These are sometimes called **customer behavioral features** or **RFM features** (Recency, Frequency, Monetary).

In [ ]:
# Compute customer-level aggregates from the entire dataset
customer_features = df.groupby("customer_id").agg(
    total_orders          = ("order_id", "count"),
    total_spend_usd       = ("final_price_usd", "sum"),
    avg_order_value       = ("final_price_usd", "mean"),
    total_items_purchased = ("quantity", "sum"),
    return_rate           = ("is_returned", "mean"),
    avg_discount_received = ("discount_pct", "mean"),
    favourite_category    = ("product_category", lambda x: x.mode()[0]),
    days_since_last_order = ("days_since_order", "min"),   # Most recent = smallest days_since
).round(2).reset_index()

print(f"Customer features shape: {customer_features.shape}")
customer_features.head(10)

In [ ]:
# Merge customer features back to the order-level data
df = df.merge(customer_features, on="customer_id", how="left", suffixes=("", "_customer"))

# Now each order row also has the customer's overall profile attached
print("Customer-level features attached to each order:")
print(df[["order_id", "customer_id", "total_orders", "avg_order_value", "return_rate"]].head(8).to_string())

---

## Section 7 - Lag and Rolling Window Features for Time Series

For sequential data, the past is a feature. Lag features make the value at time `t-1`, `t-2`, etc. available as input to predict time `t`. Rolling features summarise a window of history.

In [ ]:
# Aggregate total daily revenue for the time series example
daily = (
    df.groupby("order_date")["final_price_usd"]
    .sum()
    .reset_index()
    .rename(columns={"final_price_usd": "daily_revenue"})
    .sort_values("order_date")
)

# Lag features: revenue from 1 and 7 days ago
daily["revenue_lag_1d"]  = daily["daily_revenue"].shift(1)
daily["revenue_lag_7d"]  = daily["daily_revenue"].shift(7)

# Rolling window features
daily["revenue_roll_7d_mean"] = daily["daily_revenue"].rolling(window=7,  min_periods=1).mean().round(2)
daily["revenue_roll_7d_std"]  = daily["daily_revenue"].rolling(window=7,  min_periods=1).std().round(2)
daily["revenue_roll_30d_mean"]= daily["daily_revenue"].rolling(window=30, min_periods=1).mean().round(2)

# Percentage change from previous day
daily["revenue_pct_change"]   = daily["daily_revenue"].pct_change().round(4)

print("Daily revenue with lag and rolling features:")
print(daily.tail(10).to_string())

In [ ]:
fig, ax = plt.subplots()
ax.plot(daily["order_date"], daily["daily_revenue"],          alpha=0.4, color="#a0bce8", label="Daily revenue")
ax.plot(daily["order_date"], daily["revenue_roll_7d_mean"],   color="#2a5fc1", linewidth=1.8, label="7-day rolling mean")
ax.plot(daily["order_date"], daily["revenue_roll_30d_mean"],  color="#e86d6d", linewidth=1.8, label="30-day rolling mean")
ax.set_title("Daily revenue with rolling mean features")
ax.set_ylabel("Revenue (USD)")
ax.legend()
plt.tight_layout()
plt.savefig("output/fe_rolling_features.png", dpi=120)
plt.show()

---

## Section 8 - Target Encoding

Target encoding replaces a categorical variable with the mean of the target variable for each category. It is more informative than one-hot encoding for high-cardinality columns (e.g. a column with 50+ categories).

**Warning:** Target encoding must be done using only the training set statistics. Encoding using the full dataset leaks the target into your features.

In [ ]:
# Compute mean return rate per product category (this is a simple target encoding)
# In a real workflow, compute this on the training set only

category_return_rate = df.groupby("product_category")["is_returned"].mean().rename("category_return_rate")

print("Mean return rate per category (target encoding source):")
print(category_return_rate.round(3).to_string())

In [ ]:
# Merge the encoding back to the main DataFrame
df = df.merge(category_return_rate, on="product_category", how="left")

print("Target-encoded feature attached:")
print(df[["order_id", "product_category", "is_returned", "category_return_rate"]].head(10).to_string())

---

## Section 9 - Feature Selection: Correlation and Variance

Feature engineering creates many new columns. Before training a model, remove features that are unlikely to be useful:

1. **Low variance features** - columns that are almost constant carry no information
2. **Highly correlated features** - if two features are nearly identical, one is redundant

In [ ]:
# Select only numeric columns for the correlation analysis
numeric_df = df.select_dtypes(include="number").drop(columns=["customer_id"])

# Step 1: Drop near-zero variance features (std < 0.01 relative to mean)
variance = numeric_df.var()
low_var_cols = variance[variance < 0.01].index.tolist()
print(f"Low variance columns to consider dropping: {low_var_cols}")

In [ ]:
# Step 2: Identify highly correlated pairs (|r| > 0.90)
corr_matrix = numeric_df.corr().abs()

# Upper triangle mask to avoid duplicate pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.90
]

print("Highly correlated feature pairs (|r| > 0.90):")
for feat1, feat2, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
    print(f"  {feat1:<35} <-> {feat2:<35} r = {r:.3f}")

In [ ]:
# Correlation with the target variable (is_returned)
target_corr = numeric_df.corr()["is_returned"].drop("is_returned").sort_values(key=abs, ascending=False)

print("Feature correlation with target (is_returned):")
print(target_corr.round(3).to_string())

In [ ]:
top_features = target_corr.abs().nlargest(10)

fig, ax = plt.subplots(figsize=(9, 4))
target_corr[top_features.index].plot(
    kind="barh", ax=ax,
    color=["#e86d6d" if v > 0 else "#2a5fc1" for v in target_corr[top_features.index]]
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 10 features by correlation with is_returned")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.savefig("output/fe_feature_correlation.png", dpi=120)
plt.show()

---

## Section 10 - Building the Final Feature Set

In [ ]:
# Define which features to include in the final ML-ready dataset
# This list removes ID columns, raw dates, and redundant correlated features

feature_cols = [
    # Numeric - original
    "unit_price_usd", "quantity", "discount_pct", "final_price_usd",

    # Numeric - engineered
    "saving_usd", "price_per_unit", "log_final_price", "quantity_x_discount",

    # Category encoding
    "category_return_rate",

    # Datetime
    "order_month", "order_quarter", "is_weekend",
    "month_sin", "month_cos", "dow_sin", "dow_cos",
    "days_since_order",

    # Customer behavioral
    "total_orders", "avg_order_value", "return_rate", "avg_discount_received",
    "days_since_last_order",
]

target_col = "is_returned"

df_features = df[feature_cols + [target_col]].copy()

print(f"Final feature set: {df_features.shape[1] - 1} features + 1 target")
print(f"Rows: {len(df_features)}")
print(f"Missing values: {df_features.isnull().sum().sum()}")

df_features.head()

In [ ]:
df_features.to_csv("output/ecommerce_features.csv", index=False)
daily.to_csv("output/daily_revenue_features.csv", index=False)
customer_features.to_csv("output/customer_features.csv", index=False)

print("Saved ecommerce_features.csv")
print("Saved daily_revenue_features.csv")
print("Saved customer_features.csv")

---

## Section 11 - Summary: Feature Engineering Techniques

| Technique | Method | When to use |
|---|---|---|
| Standard scaling | `StandardScaler` | Most ML algorithms; normally distributed data |
| Min-Max scaling | `MinMaxScaler` | Neural networks; need bounded range |
| Fixed binning | `pd.cut` | When you know meaningful thresholds |
| Quantile binning | `pd.qcut` | When you want equal-sized groups |
| Log transform | `np.log1p` | Right-skewed distributions |
| Ratio features | Column division | Normalise by quantity or size |
| Interaction features | Column multiplication | Non-linear relationships between two features |
| Cyclical encoding | `sin/cos` | Month, day-of-week, hour |
| Lag features | `.shift(n)` | Predicting the next value in a sequence |
| Rolling features | `.rolling().mean()` | Smoothed history, trend |
| Target encoding | Group mean of target | High-cardinality categoricals |
| Customer aggregates | `groupby().agg()` | Transactional / e-commerce data |

---

**Practice challenge:** Create a new binary feature `is_bulk_order` that is 1 when `quantity >= 3` AND `discount_pct >= 15`. Then check whether `is_bulk_order` is more correlated with `is_returned` than either of its component features.